[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/graph_theory/07_spectral_clustering_and_gnn_applications/exercises.ipynb)

# Exercises — Topic 07: Spectral Clustering and GNN Applications

20 fully solved problems in 4 levels: Concept Check (4), Foundation (6), Applications in AI/ML & Physics (6), Challenge (4).

## Level 0 — Concept Check

### Problem L0.1: Why Plain Min-Cut Is Useless

On the unweighted path $1 - 2 - 3 - 4 - 5$, compare the bipartitions $A = \{1\}$ and $A' = \{1,2\}$ under three objectives: raw cut, RatioCut, and NCut. Which objective distinguishes them, and why does raw min-cut fail as a clustering criterion?

**Solution**

Degrees: $d_1 = d_5 = 1$, $d_2 = d_3 = d_4 = 2$; $m = 4$ and $\mathrm{vol}(V) = 2m = 8$.

**Raw cut.** Both partitions sever exactly one edge:

$$
\mathrm{cut}(A, \bar A) = 1 = \mathrm{cut}(A', \bar{A'})
$$

Indeed *every* bipartition into two intervals of the path cuts one edge, so the min-cut objective is completely indifferent — and among all bipartitions it will happily return $A = \{1\}$, peeling off a single vertex.

**RatioCut.**

$$
\mathrm{RatioCut}(A) = \frac{1}{1} + \frac{1}{4} = 1.25, \qquad \mathrm{RatioCut}(A') = \frac{1}{2} + \frac{1}{3} = 0.8\overline{3}
$$

**NCut.** $\mathrm{vol}(A) = 1$, $\mathrm{vol}(\bar A) = 7$; $\mathrm{vol}(A') = 3$, $\mathrm{vol}(\bar{A'}) = 5$:

$$
\mathrm{NCut}(A) = \frac{1}{1} + \frac{1}{7} \approx 1.143, \qquad \mathrm{NCut}(A') = \frac{1}{3} + \frac{1}{5} \approx 0.533
$$

Both balanced objectives strongly prefer $A'$; the raw cut cannot tell them apart.

$$
\boxed{\text{cut}: 1 = 1 \ \text{(tie)}; \quad \mathrm{RatioCut}: 1.25 \gt 0.833; \quad \mathrm{NCut}: 1.143 \gt 0.533}
$$

*Key takeaway:* Min-cut is polynomial-time but degenerate; dividing by $\vert A_i\vert$ or $\mathrm{vol}(A_i)$ is what makes the objective meaningful — and simultaneously what makes it NP-hard, hence the need for relaxation.

### Problem L0.2: Which Eigenvectors, and How Many?

A dataset of $n = 500$ points yields a similarity graph whose normalized Laplacian $L_{\mathrm{rw}}$ has smallest eigenvalues $0,\ 0.004,\ 0.007,\ 0.31,\ 0.36,\ 0.41,\dots$. (a) Which eigenvectors form the spectral embedding? (b) What $k$ does the eigengap heuristic suggest? (c) What is the relaxed NCut optimum for that $k$?

**Solution**

**(a)** The embedding uses the eigenvectors of the $k$ **smallest** eigenvalues of the Laplacian — not the largest. Small Laplacian eigenvalue means small Dirichlet energy, i.e. a signal that is nearly constant inside clusters. (Equivalently, they are the *largest* eigenvalues of $P = D^{-1}A$, since $L_{\mathrm{rw}} = I - P$; confusing the two operators is the single most common implementation bug.)

**(b) Eigengap heuristic.** Consecutive gaps:

| gap | $\lambda_2-\lambda_1$ | $\lambda_3-\lambda_2$ | $\lambda_4-\lambda_3$ | $\lambda_5-\lambda_4$ | $\lambda_6-\lambda_5$ |
|---|---|---|---|---|---|
| value | $0.004$ | $0.003$ | $0.303$ | $0.05$ | $0.05$ |

The largest gap follows $\lambda_3$, so $k = 3$: there are three "slow", cluster-like modes before the bulk begins.

**(c)** By the relaxation theorem, the relaxed NCut optimum is the sum of the bottom $k$ eigenvalues:

$$
\sum_{i=1}^{3}\lambda_i = 0 + 0.004 + 0.007 = 0.011
$$

which is a **lower bound** on the true discrete NCut of the best $3$-partition.

$$
\boxed{k = 3; \ \text{use } u_1,u_2,u_3 \text{ of the smallest eigenvalues}; \ \mathrm{NCut}^{\star} \ge 0.011}
$$

*Key takeaway:* Three near-zero eigenvalues means the graph is *almost* three disconnected pieces; the eigengap is a direct, computable read-out of "how many clusters are really there".

### Problem L0.3: Three Algorithms on the Star $K_{1,3}$

State what unnormalized, Shi–Malik, and Ng–Jordan–Weiss spectral clustering each diagonalize. Using $\mathrm{spec}(L(K_{1,3})) = \{0,1,1,4\}$ and $\mathrm{spec}(L_{\mathrm{sym}}) = \{0,1,1,2\}$, explain what goes wrong here, and state when all three coincide.

**Solution**

| Algorithm | Diagonalizes | Embedding of vertex $v$ |
|---|---|---|
| Unnormalized | $L = D - A$ | row $v$ of $U$ (bottom $k$ eigenvectors of $L$) |
| Shi–Malik | $Lu = \lambda D u$, i.e. $L_{\mathrm{rw}}$ | row $v$ of $U$ |
| Ng–Jordan–Weiss | $L_{\mathrm{sym}}$ | row $v$ of $U$, rescaled to unit norm |

**What goes wrong on a star.** With $k = 2$ the second eigenvector must come from the eigenvalue $1$, whose eigenspace is spanned by *leaf differences* such as $(0, 1, -1, 0)^{\top}$ (centre first). Any such vector separates one leaf from another and leaves the remaining vertices at $0$ — an arbitrary split with no cluster meaning. The honest diagnosis is that a star has **no** two-cluster structure: $\lambda_2 = 1$ is far from $0$, so there is no eigengap and no near-disconnection to detect.

**When the three agree.** On a $d$-regular graph $D = dI$, so

$$
L_{\mathrm{sym}} = L_{\mathrm{rw}} = \tfrac{1}{d}L
$$

All three diagonalize the same matrix up to the scalar $1/d$, share eigenvectors, and the NJW row normalization is a no-op up to a constant. Differences appear only through degree heterogeneity.

$$
\boxed{\text{Regular graph} \Rightarrow \text{all three identical}; \ \text{star} \Rightarrow \text{degenerate } \lambda_2 = 1, \text{ no cluster structure}}
$$

*Key takeaway:* Always look at the eigenvalues before trusting the eigenvectors; a large $\lambda_2$ or a degenerate eigenspace means the algorithm is being asked a question the graph cannot answer.

### Problem L0.4: Cost and Parameter Count of a Two-Layer GCN

For the Cora citation graph ($n = 2708$ nodes, $m = 5429$ undirected edges, $f_0 = 1433$ input features, hidden width $f_1 = 16$, $f_2 = 7$ classes), compute (a) the number of weight parameters, (b) the number of nonzeros in $\hat{S}$, and (c) the dominant cost of a forward pass.

**Solution**

**(a) Parameters.** Only the $W^{(l)}$ are learned:

$$
f_0 f_1 + f_1 f_2 = 1433 \cdot 16 + 16 \cdot 7 = 22{,}928 + 112 = 23{,}040
$$

(about $23$k — two orders of magnitude smaller than a comparable MLP on the same features with a wide hidden layer).

**(b) Nonzeros of $\hat{S} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$.** $\tilde{A} = A + I$ has $2m$ off-diagonal nonzeros plus $n$ self-loops:

$$
\mathrm{nnz}(\hat S) = 2m + n = 10{,}858 + 2708 = 13{,}566
$$

**(c) Forward cost.** Layer 1 is $\hat{S}(XW^{(0)})$, computed right-to-left:

- dense feature map $XW^{(0)}$: $n f_0 f_1 = 2708 \cdot 1433 \cdot 16 \approx 6.2 \times 10^{7}$ multiply–adds;
- sparse propagation $\hat{S}\,(\cdot)$: $\mathrm{nnz}(\hat S)\cdot f_1 = 13{,}566 \cdot 16 \approx 2.2\times 10^{5}$.

Layer 2 is negligible ($n f_1 f_2 \approx 3\times 10^{5}$). So the **dense feature transform dominates by roughly $300\times$** — the graph part of a GCN is essentially free.

$$
\boxed{23{,}040 \text{ parameters}; \ \mathrm{nnz}(\hat S) = 13{,}566; \ \text{cost} \approx 6.2\times10^{7} \text{ dominated by } XW}
$$

*Key takeaway:* GCNs are cheap because propagation is sparse and never spectral; always multiply $XW$ first, then propagate — the reverse order costs $f_0/f_1$ times more.

## Level 1 — Foundation

### Problem L1.1: Verifying the RatioCut Identity

Let $G$ be the barbell: triangles $\{1,2,3\}$ and $\{4,5,6\}$ joined by the bridge $\{3,4\}$. For $A = \{1,2,3\}$, construct the vector $f$ of the RatioCut relaxation and verify $f^{\top}Lf = n \cdot \mathrm{RatioCut}(A,\bar A)$ by direct computation.

**Solution**

Here $n = 6$, $\vert A\vert = \vert \bar A\vert = 3$, so

$$
f_v = \sqrt{\vert \bar A\vert/\vert A\vert} = 1 \ \ (v \in A), \qquad f_v = -\sqrt{\vert A\vert/\vert \bar A\vert} = -1 \ \ (v \in \bar A)
$$

**Checks on $f$.** $\sum_v f_v = 3 - 3 = 0$ ✓ so $f \perp \mathbf{1}$; and $\Vert f\Vert^2 = 6 = n$ ✓.

**Energy.** Only edges with endpoints on opposite sides contribute; the sole such edge is the bridge $\{3,4\}$, with difference $1 - (-1) = 2$:

$$
f^{\top}Lf = \sum_{\{u,v\}\in E}(f_u - f_v)^2 = 2^2 = 4
$$

**Objective.** $\mathrm{cut}(A,\bar A) = 1$, so

$$
\mathrm{RatioCut}(A,\bar A) = \frac{1}{3} + \frac{1}{3} = \frac{2}{3}, \qquad n\cdot\mathrm{RatioCut} = 6\cdot\frac{2}{3} = 4 \ \checkmark
$$

$$
\boxed{f^{\top}Lf = 4 = n\,\mathrm{RatioCut}(A,\bar A)}
$$

*Key takeaway:* The clever normalization of $f$ exists precisely so that a *combinatorial* objective becomes a *quadratic form* — after which linear algebra, not combinatorics, takes over.

### Problem L1.2: How Loose Is the Relaxation?

For the same barbell, the Laplacian spectrum is $\{0,\ \frac{5-\sqrt{17}}{2},\ 3,3,3,\ \frac{5+\sqrt{17}}{2}\}$ and the generalized spectrum of $Lu = \lambda Du$ is $\{0,\ \frac{11-\sqrt{73}}{12},\ 1.1667,\ 1.5,\ 1.5,\ 1.6287\}$. Compare the relaxed lower bounds with the true RatioCut and NCut of the partition $\{1,2,3\}$ versus $\{4,5,6\}$.

**Solution**

**True values.** With $\mathrm{cut} = 1$, $\vert A\vert = 3$, $\mathrm{vol}(A) = 2+2+3 = 7$ (and the same for $\bar A$ by symmetry):

$$
\mathrm{RatioCut} = \frac13 + \frac13 = \frac23 \approx 0.6667, \qquad \mathrm{NCut} = \frac17 + \frac17 = \frac27 \approx 0.2857
$$

**Relaxed bounds.**

$$
\lambda_2(L) = \frac{5-\sqrt{17}}{2} \approx 0.4384 \ \le \ \mathrm{RatioCut} \approx 0.6667 \ \checkmark
$$

$$
\lambda_2(L_{\mathrm{rw}}) = \frac{11-\sqrt{73}}{12} \approx 0.2047 \ \le \ \mathrm{NCut} \approx 0.2857 \ \checkmark
$$

**Gaps.** RatioCut relaxation is loose by a factor $0.6667/0.4384 \approx 1.52$; NCut by $0.2857/0.2047 \approx 1.40$. Both are modest here because the barbell really *is* two clusters — the relaxation is tight exactly when the structure is clean.

**Cheeger cross-check.** $h(G) = \mathrm{cut}/\mathrm{vol}(A) = 1/7 \approx 0.1429$, and

$$
\frac{h^2}{2} = 0.0102 \ \le \ \lambda_2^{\mathrm{sym}} = 0.2047 \ \le \ 2h = 0.2857 \ \checkmark
$$

(the upper Cheeger bound is exactly $\mathrm{NCut}$ here, since the two sides have equal volume).

$$
\boxed{\lambda_2(L) = 0.4384 \le \tfrac23; \quad \lambda_2(L_{\mathrm{rw}}) = 0.2047 \le \tfrac27; \quad \tfrac{h^2}{2} \le \lambda_2^{\mathrm{sym}} \le 2h \ \checkmark}
$$

*Key takeaway:* Relaxed eigenvalues are always *lower bounds*; their ratio to the achieved discrete objective is a free certificate of how close to optimal your partition is.

### Problem L1.3: Ky Fan for $k = 3$

Using the barbell spectrum $\{0,\ 0.4384,\ 3,\ 3,\ 3,\ 4.5616\}$, compute the relaxed RatioCut optimum for $k = 3$ and compare with the best true $3$-way partition, which is $\{1,2,3\},\{4,5\},\{6\}$.

**Solution**

**Relaxed value.** By Ky Fan's theorem,

$$
\min_{H^{\top}H = I_3}\operatorname{tr}(H^{\top}LH) = \lambda_1 + \lambda_2 + \lambda_3 = 0 + 0.4384 + 3 = 3.4384
$$

attained by any orthonormal basis of the bottom three eigenvectors.

**True value.** For $\{1,2,3\},\{4,5\},\{6\}$: the cuts are $\mathrm{cut}(\{1,2,3\}) = 1$ (the bridge), $\mathrm{cut}(\{4,5\}) = 3$ (edges $34$, $46$, $56$), $\mathrm{cut}(\{6\}) = 2$ (edges $46$, $56$):

$$
\mathrm{RatioCut} = \frac{1}{3} + \frac{3}{2} + \frac{2}{1} = \frac{23}{6} \approx 3.8333
$$

(Exhaustive enumeration over all $3$-partitions of the $6$ vertices confirms $23/6$ is optimal.)

**Gap.** $3.8333 / 3.4384 \approx 1.115$ — an $11.5\%$ relaxation gap.

Note *why* the third eigenvalue is already $3$: the barbell has only two natural clusters, so forcing a third piece must break a triangle, which is expensive. The jump $\lambda_2 = 0.44 \to \lambda_3 = 3$ is the eigengap saying "$k = 2$", and the relaxed value for $k=3$ inherits that penalty.

$$
\boxed{\text{relaxed } \sum_{i\le3}\lambda_i = 3.4384 \ \le \ \mathrm{RatioCut}^{\star} = \tfrac{23}{6} \approx 3.8333}
$$

*Key takeaway:* Ky Fan turns "choose $k$ balanced clusters" into "add up the $k$ smallest eigenvalues"; the size of $\lambda_{k}$ relative to $\lambda_{k-1}$ tells you whether that $k$ was a sensible request.

### Problem L1.4: NCut as an Escape Probability

Verify the identity $\mathrm{NCut}(A,\bar A) = \Pr[X_1 \in \bar A \mid X_0 \in A] + \Pr[X_1 \in A \mid X_0 \in \bar A]$ on the barbell with $A = \{1,2,3\}$, by computing the escape probability directly from the walk.

**Solution**

**Stationary distribution.** $\mathrm{vol}(V) = 2m = 14$, so $\pi = (2,2,3,3,2,2)/14$.

**Direct escape computation.** Conditioned on $X_0 \in A$, the walk starts at $v \in A$ with probability $\pi_v/\pi(A) = d_v/\mathrm{vol}(A)$:

$$
\Pr[X_0 = 1 \mid X_0 \in A] = \tfrac{2}{7}, \quad \Pr[X_0 = 2 \mid \cdot] = \tfrac{2}{7}, \quad \Pr[X_0 = 3 \mid \cdot] = \tfrac{3}{7}
$$

From vertices $1$ and $2$ every neighbour is inside $A$, so escape is impossible. From vertex $3$ (degree $3$: neighbours $1, 2, 4$) exactly one neighbour lies outside, so the escape probability is $\tfrac13$. Hence

$$
\Pr[X_1 \in \bar A \mid X_0 \in A] = \tfrac{2}{7}\cdot 0 + \tfrac{2}{7}\cdot 0 + \tfrac{3}{7}\cdot\tfrac13 = \tfrac{1}{7}
$$

**Formula check.** $\mathrm{cut}(A,\bar A)/\mathrm{vol}(A) = 1/7$ ✓ — the degree $d_3 = 3$ cancels between the stationary weight $3/7$ and the transition probability $1/3$, which is the entire mechanism of the theorem.

By symmetry the reverse direction is also $1/7$, so

$$
\mathrm{NCut} = \tfrac17 + \tfrac17 = \tfrac27 \approx 0.2857
$$

matching the direct combinatorial value from Problem L1.2.

$$
\boxed{\Pr[\text{escape } A] = \tfrac17 = \frac{\mathrm{cut}(A,\bar A)}{\mathrm{vol}(A)}, \qquad \mathrm{NCut} = \tfrac27}
$$

*Key takeaway:* NCut clusters are metastable sets of the random walk — this is why NCut, and not RatioCut, is the objective with a probabilistic meaning, and why it transfers directly to Markov state models in chemistry and physics.

### Problem L1.5: Sweep Cuts of the Fiedler Vector on $P_6$

The path $P_6$ has $\lambda_2 = 2 - \sqrt{3} \approx 0.2679$ with Fiedler vector $x_v = \cos\frac{\pi(2v-1)}{12}$. Evaluate all sweep cuts, find the best one, and check it against the Cheeger sandwich given $\lambda_2^{\mathrm{sym}}(P_6) = 0.1910$.

**Solution**

**Fiedler entries** ($v = 1,\dots,6$), already in decreasing order:

| $v$ | $1$ | $2$ | $3$ | $4$ | $5$ | $6$ |
|---|---|---|---|---|---|---|
| $x_v$ | $0.966$ | $0.707$ | $0.259$ | $-0.259$ | $-0.707$ | $-0.966$ |

**Sweep cuts.** Take prefixes $S_t = \{1,\dots,t\}$ and evaluate the conductance $\mathrm{cut}(S)/\mathrm{vol}(S)$; degrees are $1,2,2,2,2,1$ and $\mathrm{vol}(V) = 10$:

| $S$ | $\mathrm{cut}$ | $\mathrm{vol}(S)$ | conductance |
|---|---|---|---|
| $\{1\}$ | $1$ | $1$ | $1.000$ |
| $\{1,2\}$ | $1$ | $3$ | $0.333$ |
| $\{1,2,3\}$ | $1$ | $5$ | $0.200$ |

The best sweep cut is $S = \{1,2,3\}$ with conductance $0.2$ — which is exactly the **sign cut** of the Fiedler vector, and equals $h(P_6)$.

**Cheeger sandwich.** With $h = 0.2$ and $\lambda_2^{\mathrm{sym}} = 0.1910$:

$$
\frac{h^2}{2} = 0.02 \ \le \ 0.1910 \ \le \ 2h = 0.4 \ \checkmark
$$

$$
\boxed{\text{best sweep cut } \{1,2,3\}, \ h(P_6) = 0.2; \quad 0.02 \le \lambda_2^{\mathrm{sym}} = 0.191 \le 0.4}
$$

*Key takeaway:* Rounding is not an afterthought — sweeping over all $n-1$ thresholds costs $O(m)$ total and is the step covered by the Cheeger guarantee; the naive sign cut happens to be optimal here but is not in general.

### Problem L1.6: Translating Between $L_{\mathrm{sym}}$ and $L_{\mathrm{rw}}$

Prove that $L_{\mathrm{sym}}t = \mu t \iff L_{\mathrm{rw}}u = \mu u$ with $u = D^{-1/2}t$, and verify explicitly on the star $K_{1,3}$ for $\mu = 2$.

**Solution**

**Proof.** By definition $L_{\mathrm{rw}} = D^{-1}L = D^{-1/2}\big(D^{-1/2}LD^{-1/2}\big)D^{1/2} = D^{-1/2}L_{\mathrm{sym}}D^{1/2}$. Hence, with $u = D^{-1/2}t$,

$$
L_{\mathrm{rw}}u = D^{-1/2}L_{\mathrm{sym}}D^{1/2}D^{-1/2}t = D^{-1/2}L_{\mathrm{sym}}t = D^{-1/2}(\mu t) = \mu\,D^{-1/2}t = \mu u
$$

and the argument reverses. The two operators are **similar**: identical eigenvalues, eigenvectors related by $D^{\pm 1/2}$. Equivalently, both are restatements of the generalized problem $Lu = \mu D u$.

**Verification on $K_{1,3}$** (centre $0$, leaves $1,2,3$; $D = \mathrm{diag}(3,1,1,1)$). Take

$$
u = (1,-1,-1,-1)^{\top}, \qquad t = D^{1/2}u = (\sqrt{3},-1,-1,-1)^{\top}
$$

*Check $L_{\mathrm{rw}}u = 2u$*, using $(L_{\mathrm{rw}}u)_v = u_v - \frac{1}{d_v}\sum_{w \sim v}u_w$:

- centre: $1 - \frac13(-1-1-1) = 1 + 1 = 2 = 2u_0$ ✓
- each leaf: $-1 - \frac11(1) = -2 = 2u_{\text{leaf}}$ ✓

*Check $L_{\mathrm{sym}}t = 2t$*, using $(L_{\mathrm{sym}}t)_v = t_v - \sum_{w\sim v}\frac{t_w}{\sqrt{d_vd_w}}$:

- centre: $\sqrt3 - 3\cdot\frac{-1}{\sqrt{3}} = \sqrt3 + \sqrt3 = 2\sqrt3$ ✓
- each leaf: $-1 - \frac{\sqrt3}{\sqrt3} = -2$ ✓

The eigenvalue $2$ appears because the star is bipartite (Topic 06), and $u$ is exactly $+1$ on the centre, $-1$ on the leaf side.

$$
\boxed{L_{\mathrm{rw}} = D^{-1/2}L_{\mathrm{sym}}D^{1/2}: \ \text{same spectrum}, \ u = D^{-1/2}t}
$$

*Key takeaway:* Shi–Malik and Ng–Jordan–Weiss solve the same eigenproblem in different coordinates; the $D^{\pm1/2}$ factor is exactly the discrepancy that NJW's row normalization is designed to undo.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: The Gaussian Bandwidth Decides Everything

Six one-dimensional points sit at $0,\ 0.1,\ 0.2$ and $5.0,\ 5.1,\ 5.2$. Build a fully connected similarity graph with $w_{ij} = \exp\!\big(-\Vert x_i-x_j\Vert^2/(2\sigma^2)\big)$ and compute the NCut of the obvious bipartition for (a) $\sigma = 0.5$ and (b) $\sigma = 5$. What do you conclude?

**Solution**

Let $A = \{x_1,x_2,x_3\}$ (the left group), $\bar A$ the right group.

**(a) $\sigma = 0.5$, so $2\sigma^2 = 0.5$ and $w = e^{-2d^2}$.**

Within-group distances are $0.1$ and $0.2$: $w = e^{-0.02} = 0.980$ and $e^{-0.08} = 0.923$. Cross-group distances are $\ge 4.8$: $w \le e^{-2(4.8)^2} = e^{-46.1} \approx 10^{-20}$.

$$
\mathrm{cut}(A,\bar A) \approx 9 \times 10^{-20}, \qquad \mathrm{vol}(A) \approx 2(0.980+0.923+0.980) = 5.767
$$

$$
\mathrm{NCut} \approx 2\cdot\frac{9\times10^{-20}}{5.767} \approx 3\times 10^{-20}
$$

The graph is *numerically disconnected*: $\lambda_2 \approx 0$, the Fiedler vector is (to machine precision) a group indicator, and clustering is exact.

**(b) $\sigma = 5$, so $2\sigma^2 = 50$.**

Within-group weights are now $\approx e^{-0.0002} \approx 0.9998$; cross-group weights are $e^{-d^2/50}$ with $d \approx 5$, i.e. $\approx 0.58$–$0.63$. Summing the nine cross pairs gives $\mathrm{cut} \approx 5.46$, and

$$
\mathrm{vol}(A) = 2(0.9998+0.9992+0.9998) + 5.46 \approx 11.46 \quad \Longrightarrow \quad \mathrm{NCut} \approx 2\cdot\frac{5.46}{11.46} \approx 0.95
$$

Since $\mathrm{NCut} \le 2$ always, a value near $1$ means the "clusters" are barely distinguishable — the kernel has smeared them into one blob.

$$
\boxed{\sigma = 0.5:\ \mathrm{NCut}\approx 3\times10^{-20}\ \text{(perfect)}; \qquad \sigma = 5:\ \mathrm{NCut}\approx 0.95\ \text{(no structure)}}
$$

*Key takeaway:* Nineteen orders of magnitude separate a good bandwidth from a bad one. Graph construction, not the eigensolver, is where spectral clustering succeeds or fails — hence local scaling ($\sigma_i$ = distance to the $7$-th neighbour) or kNN graphs in practice.

### Problem L2.2: Normalized Cuts on a One-Dimensional "Image"

Six pixels have intensities $10, 11, 12, 50, 51, 52$. Connect adjacent pixels with $w_{i,i+1} = \exp\!\big(-(I_i - I_{i+1})^2/(2\cdot 5^2)\big)$. Compute the NCut of the segmentation at the intensity boundary and compare it with peeling off the first pixel.

**Solution**

**Edge weights.** Adjacent intensity differences are $1,1,38,1,1$, and $2\sigma^2 = 50$:

$$
w = e^{-1/50} = 0.980 \ \text{(four times)}, \qquad w_{34} = e^{-38^2/50} = e^{-28.9} \approx 2.8\times10^{-13}
$$

Total edge weight $= 4(0.980) + 2.8\times10^{-13} \approx 3.920$, so $\mathrm{vol}(V) \approx 7.840$.

**Segmentation at the boundary**, $A = \{1,2,3\}$: $\mathrm{cut} = w_{34} \approx 2.8\times10^{-13}$, $\mathrm{vol}(A) = 0.980 + 1.960 + 0.980 = 3.920$ (and the same for $\bar A$):

$$
\mathrm{NCut} \approx 2\cdot\frac{2.8\times10^{-13}}{3.920} \approx 1.4\times10^{-13}
$$

**Peeling the first pixel**, $A' = \{1\}$: $\mathrm{cut} = 0.980$, $\mathrm{vol}(A') = 0.980$, $\mathrm{vol}(\bar{A'}) = 6.860$:

$$
\mathrm{NCut} = \frac{0.980}{0.980} + \frac{0.980}{6.860} = 1 + 0.143 = 1.143
$$

The intensity boundary wins by thirteen orders of magnitude. Note that plain min-cut would *also* pick the boundary here (its cut weight is smallest), but on a real 2-D image, where the boundary is long and interior pixels are isolated by noise, only the normalized objective avoids peeling off single pixels — precisely Shi & Malik's motivating observation.

$$
\boxed{\mathrm{NCut}(\text{boundary}) \approx 1.4\times10^{-13} \ \lll \ \mathrm{NCut}(\{1\}) = 1.143}
$$

*Key takeaway:* Turning intensity differences into exponential weights makes strong edges *exponentially* cheap to cut; segmentation quality is dominated by the choice of $\sigma$ relative to the true contrast, exactly as in Problem L2.1.

### Problem L2.3: One GCN Layer by Hand

For the path $1 - 2 - 3$ with a single input channel $x = (1, 0, -1)^{\top}$ and $W = 1$, construct $\tilde{A}$, $\tilde{D}$, and $\hat{S} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$, then compute one propagation step and the effect of a ReLU.

**Solution**

**Self-looped adjacency and degrees.**

$$
\tilde{A} = A + I = \begin{pmatrix} 1 & 1 & 0 \\ 1 & 1 & 1 \\ 0 & 1 & 1\end{pmatrix}, \qquad \tilde{d} = (2, 3, 2)
$$

**Propagation matrix.** $\hat{S}_{uv} = \tilde{A}_{uv}/\sqrt{\tilde{d}_u\tilde{d}_v}$:

$$
\hat{S} = \begin{pmatrix} \tfrac12 & \tfrac{1}{\sqrt6} & 0 \\ \tfrac{1}{\sqrt6} & \tfrac13 & \tfrac{1}{\sqrt6} \\ 0 & \tfrac{1}{\sqrt6} & \tfrac12 \end{pmatrix} \approx \begin{pmatrix} 0.500 & 0.408 & 0 \\ 0.408 & 0.333 & 0.408 \\ 0 & 0.408 & 0.500\end{pmatrix}
$$

Note the rows do **not** sum to $1$ ($0.908, 1.150, 0.908$): the symmetric normalization is not a stochastic matrix — the row-stochastic alternative $\tilde{D}^{-1}\tilde{A}$ is mean aggregation instead.

**One layer.**

$$
\hat{S}x = \begin{pmatrix} 0.5(1) + 0.408(0) + 0 \\ 0.408(1) + 0.333(0) + 0.408(-1) \\ 0 + 0.408(0) + 0.5(-1)\end{pmatrix} = \begin{pmatrix} 0.5 \\ 0 \\ -0.5\end{pmatrix}
$$

**After ReLU.** $H^{(1)} = \max(\hat{S}x, 0) = (0.5,\ 0,\ 0)^{\top}$.

**Smoothing check.** The input has range $[-1,1]$ and total variation $\sum_{u\sim v}\vert x_u - x_v\vert = 2$; the output has range $[-0.5, 0.5]$ and total variation $1$. One layer halved the signal's roughness — this specific $x$ is an eigenvector of $\hat{S}$ with eigenvalue exactly $\frac12$ (see Problem L2.4).

$$
\boxed{\hat{S}x = (0.5,\ 0,\ -0.5)^{\top}, \qquad \sigma(\hat{S}x) = (0.5,\ 0,\ 0)^{\top}}
$$

*Key takeaway:* A GCN layer is a weighted neighbourhood average followed by a learned linear map; the "convolution" is fully described by the sparse matrix $\hat{S}$, which you can and should print out when debugging.

### Problem L2.4: Over-Smoothing Rate on the Same Graph

For the $\hat{S}$ of Problem L2.3, find all eigenvalues, identify the limit of $\hat{S}^{\,l}$, and determine how many layers reduce the non-constant signal by a factor $1000$.

**Solution**

**Top eigenpair.** $\hat{S}\,\tilde{D}^{1/2}\mathbf{1} = \tilde{D}^{-1/2}\tilde{A}\mathbf{1} = \tilde{D}^{1/2}\mathbf{1}$, so $\mu_1 = 1$ with

$$
v_1 = \frac{(\sqrt2, \sqrt3, \sqrt2)^{\top}}{\sqrt{7}}
$$

**Second eigenpair.** By the reflection symmetry $1 \leftrightarrow 3$, try $x = (1,0,-1)^{\top}$:

$$
\hat{S}x = (0.5,\ 0,\ -0.5)^{\top} = \tfrac12 x \quad \Longrightarrow \quad \mu_2 = \tfrac12
$$

**Third eigenvalue by the trace.** $\operatorname{tr}\hat S = \frac12+\frac13+\frac12 = \frac43$, so $\mu_3 = \frac43 - 1 - \frac12 = -\frac16$.

$$
\mathrm{spec}(\hat S) = \Big\{1,\ \tfrac12,\ -\tfrac16\Big\} \subset (-1, 1] \ \checkmark
$$

**Limit and rate.**

$$
\hat{S}^{\,l} = \sum_i \mu_i^{\,l}v_iv_i^{\top} \ \longrightarrow \ v_1v_1^{\top} = \frac{1}{7}\begin{pmatrix} 2 & \sqrt6 & 2 \\ \sqrt6 & 3 & \sqrt6 \\ 2 & \sqrt6 & 2\end{pmatrix}
$$

with error $\max(\vert\mu_2\vert,\vert\mu_3\vert)^{\,l} = (1/2)^{l}$. For our $x$ (which is orthogonal to $v_1$, since $v_1^{\top}x = (\sqrt2-\sqrt2)/\sqrt7 = 0$), $\hat{S}^{\,l}x = 2^{-l}x \to 0$ exactly.

Reducing by $1000$ needs $2^{-l} \le 10^{-3}$, i.e.

$$
l \ge \frac{\ln 1000}{\ln 2} \approx 9.97 \quad \Longrightarrow \quad l = 10 \ \text{layers}
$$

$$
\boxed{\mathrm{spec}(\hat S) = \{1, \tfrac12, -\tfrac16\}, \quad \hat{S}^{\,l} \to v_1v_1^{\top} \ \text{ at rate } 2^{-l}, \quad l = 10 \text{ kills } 10^{3}}
$$

*Key takeaway:* Over-smoothing is geometric decay of every non-constant mode; the surviving direction $v_1 \propto \sqrt{\tilde{d}_v}$ carries only degree information, which is why deep vanilla GCNs converge to a degree-based classifier.

### Problem L2.5: Spectral Community Detection in the Stochastic Block Model

An SBM has two blocks of $n/2$ vertices, within-block edge probability $p$ and across-block probability $q \lt p$. Compute the Laplacian spectrum of the *expected* graph, identify the Fiedler vector, and evaluate the eigengap for $n = 1000$, $p = 0.05$, $q = 0.01$.

**Solution**

Work with $\bar{A} = \mathbb{E}[A]$, which is $p$ within blocks and $q$ across (ignoring the $O(1)$ diagonal correction). Every expected degree is

$$
\bar{d} = p\Big(\frac n2 - 1\Big) + q\,\frac n2 \approx \frac n2 (p + q)
$$

**Community mode.** Let $u = \mathbf{1}_{\text{block }1} - \mathbf{1}_{\text{block }2}$. For $i$ in block $1$,

$$
(\bar{A}u)_i \approx p\,\frac n2 - q\,\frac n2 = \frac n2 (p-q)
$$

so $\bar{A}u = \frac n2(p-q)u$ and

$$
\bar{L}u = \bar{d}u - \bar{A}u = \Big[\frac n2(p+q) - \frac n2 (p-q)\Big]u = n q\, u
$$

**Bulk modes.** For $x$ summing to zero *within each block*, $\bar{A}x = 0$, hence $\bar{L}x = \bar{d}\,x = \frac n2 (p+q)x$, an eigenvalue of multiplicity $n - 2$. Together with $\bar{L}\mathbf{1} = 0$:

$$
\mathrm{spec}(\bar{L}) = \Big\{0, \ \ nq \ (\text{once}), \ \ \tfrac n2 (p+q) \ (\text{multiplicity } n-2)\Big\}
$$

Since $q \lt p$ we have $nq \lt \frac n2(p+q)$, so $\lambda_2 = nq$ and the **Fiedler vector is exactly the block indicator** $u$ — spectral clustering recovers the communities perfectly in expectation.

**Numbers** ($n = 1000$, $p = 0.05$, $q = 0.01$):

$$
\lambda_2 = 1000(0.01) = 10, \qquad \lambda_3 = 500(0.06) = 30, \qquad \text{eigengap } \lambda_3 - \lambda_2 = 20
$$

$$
\boxed{\lambda_2(\bar L) = nq = 10 \ \lt \ \lambda_3 = \tfrac n2(p+q) = 30; \ \text{Fiedler vector} = \text{block indicator}}
$$

*Key takeaway:* In expectation the SBM is a rank-two perturbation of a constant matrix, so its Fiedler vector is exactly the labels; real graphs add sampling noise, and Davis–Kahan says recovery survives as long as that noise is small relative to the eigengap $\frac n2(p+q) - nq = \frac n2(p - q)$.

### Problem L2.6: Depth Versus Receptive Field

A GCN is applied to a citation graph of diameter $D = 10$ whose propagation matrix has $\vert\mu_2\vert = 0.9$. (a) How many layers are needed for information to travel between the two most distant nodes? (b) How much of the discriminative signal survives that depth? (c) What is the practical resolution?

**Solution**

**(a) Receptive field.** One layer aggregates $1$-hop neighbours, so $l$ layers give an $l$-hop receptive field. Covering the diameter requires

$$
l \ge D = 10 \ \text{layers}
$$

**(b) Signal decay.** By the over-smoothing theorem, the component of the representations orthogonal to $v_1 \propto \tilde{D}^{1/2}\mathbf{1}$ is multiplied by at most $\vert\mu_2\vert$ per layer:

$$
\text{surviving fraction} \le 0.9^{10} \approx 0.349
$$

and the *relative* discriminative content decays further because $v_1$'s component is untouched. At $l = 20$ layers it is $0.9^{20} \approx 0.12$; at $l = 50$, $0.005$.

**(c) The tension.** Depth buys reach and destroys contrast simultaneously — an $l$-hop receptive field costs a factor $\vert\mu_2\vert^{\,l}$ in signal. The standard resolutions are:

- keep $l = 2$–$3$ and accept a local receptive field (usually enough: homophilous graphs need only $1$–$2$ hops);
- add **residual / initial connections** (GCNII: $H^{(l+1)} = \sigma\big(((1-\alpha)\hat{S}H^{(l)} + \alpha H^{(0)})((1-\beta)I + \beta W)\big)$), which keeps a fixed fraction of the input and provably prevents collapse;
- use **jumping knowledge** (concatenate all layer outputs) so the classifier can pick the right hop count per node;
- **decouple** propagation from transformation (APPNP / SGC): propagate with a personalized-PageRank operator $\alpha(I - (1-\alpha)\hat{S})^{-1}$, whose filter never collapses to rank one.

$$
\boxed{l \ge D = 10 \ \text{for full reach, but only } 0.9^{10} \approx 35\% \ \text{of the non-constant signal survives}}
$$

*Key takeaway:* Receptive field grows linearly in depth while discriminative signal decays geometrically — which is why GNN architecture research is largely the search for propagation operators whose powers do *not* converge to rank one.

## Level 3 — Challenge

### Problem L3.1: Spectral Clustering Is Exact on Disconnected Graphs

Prove: if $G$ has exactly $k$ connected components $A_1,\dots,A_k$, then the eigenvalue $0$ of $L_{\mathrm{rw}}$ has multiplicity $k$, the spectral embedding maps every vertex of $A_i$ to a single point $y^{(i)} \in \mathbb{R}^k$, and the $k$ points are distinct — so $k$-means recovers the components exactly. Then explain what survives when the components are weakly connected.

**Solution**

**Step 1 — the eigenspace.** Since $D$ is invertible, $L_{\mathrm{rw}}u = 0 \iff Lu = 0$. By Topic 06's kernel theorem, $\ker L = \mathrm{span}\{\mathbf{1}_{A_1},\dots,\mathbf{1}_{A_k}\}$, which has dimension exactly $k$. So the multiplicity of $0$ is $k$ and the eigenspace is the space of functions constant on components.

**Step 2 — the embedding is component-constant.** Let $U = [u_1 \cdots u_k]$ be *any* basis of this eigenspace arranged as columns. Each $u_j$ is constant on components, so there is a matrix $M \in \mathbb{R}^{k\times k}$ with

$$
U = \big[\mathbf{1}_{A_1} \ \cdots \ \mathbf{1}_{A_k}\big]\,M, \qquad M_{ij} = \text{value of } u_j \text{ on } A_i
$$

The row of $U$ belonging to $v \in A_i$ is therefore $y_v = M_{i,\cdot}$ — it depends only on $i$, not on $v$. So all vertices of $A_i$ collapse to a single embedded point $y^{(i)} = M_{i,\cdot}$.

**Step 3 — the $k$ points are distinct.** Because $u_1,\dots,u_k$ are linearly independent and the indicators $\mathbf{1}_{A_i}$ form a basis of the same space, $M$ is invertible. An invertible matrix has linearly independent — in particular pairwise distinct — rows. Hence $y^{(1)},\dots,y^{(k)}$ are $k$ distinct points.

**Step 4 — $k$-means succeeds.** The embedded data set consists of exactly $k$ distinct points, each with multiplicity $\vert A_i\vert$. The $k$-means objective is $0$ if and only if the centroids are those $k$ points and each group is one cluster; any other assignment has strictly positive cost. So the global optimum recovers the components exactly. $\blacksquare$

$$
\boxed{k \text{ components} \Rightarrow \text{embedding} = k \text{ distinct points} \Rightarrow k\text{-means is exact}}
$$

**Weak connections (the perturbation argument).** Add a few light edges joining the components. Then $L_{\mathrm{rw}}$ becomes $L_{\mathrm{rw}}^{(0)} + E$ with $\Vert E\Vert$ small, the $k$ zero eigenvalues split into $0 = \lambda_1 \le \cdots \le \lambda_k \ll \lambda_{k+1}$, and the **Davis–Kahan theorem** bounds the rotation between the perturbed and unperturbed invariant subspaces by

$$
\sin\Theta \le \frac{\Vert E\Vert}{\lambda_{k+1} - \lambda_k}
$$

So the embedded points no longer coincide but stay in $k$ tight blobs whose spread is controlled by $\Vert E\Vert$ divided by the **eigengap**. That single inequality is the entire theoretical justification of spectral clustering on real data — and it explains why the eigengap heuristic is not merely a rule of thumb.

*Key takeaway:* The ideal case is exactly solvable, and everything in practice is a perturbation of it; whenever spectral clustering fails, the diagnosis is either "no eigengap" or "perturbation too large relative to the gap".

### Problem L3.2: What the Renormalization Trick Actually Does

Compare the spectra of $S_0 = I + D^{-1/2}AD^{-1/2}$ and $\hat{S} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ (with $\tilde{A} = A+I$). Prove the containments $\mathrm{spec}(S_0)\subseteq[0,2]$ and $\mathrm{spec}(\hat S)\subseteq(-1,1]$, and illustrate on $K_2$.

**Solution**

**Step 1 — $S_0$ in terms of $L_{\mathrm{sym}}$.** By definition $L_{\mathrm{sym}} = I - D^{-1/2}AD^{-1/2}$, so

$$
S_0 = I + D^{-1/2}AD^{-1/2} = 2I - L_{\mathrm{sym}}
$$

By Topic 06, $\mathrm{spec}(L_{\mathrm{sym}})\subseteq[0,2]$, hence $\mathrm{spec}(S_0) = \{2 - \lambda\} \subseteq [0,2]$. The **top eigenvalue is exactly $2$** (attained at $\lambda = 0$, which always exists). Therefore repeated application multiplies the smoothest component by $2$ each time:

$$
\Vert S_0^{\,l}\Vert_2 = 2^{\,l} \longrightarrow \infty
$$

Stacking such layers makes activations blow up exponentially unless the learned weights fight the operator — a real optimization pathology, and the reason Kipf & Welling replaced $S_0$.

**Step 2 — $\hat{S}$ in terms of the self-looped Laplacian.** Let $\tilde{L}_{\mathrm{sym}}$ be the normalized Laplacian of the graph $\tilde{G}$ with a self-loop at every vertex. Then $\hat{S} = I - \tilde{L}_{\mathrm{sym}}$ and $\mathrm{spec}(\tilde{L}_{\mathrm{sym}})\subseteq[0,2]$ gives $\mathrm{spec}(\hat S)\subseteq[-1,1]$ immediately.

*The endpoint $-1$ is excluded.* By the bipartiteness theorem (Topic 06), $2 \in \mathrm{spec}(\tilde{L}_{\mathrm{sym}})$ iff some component of $\tilde G$ is bipartite. But every vertex of $\tilde{G}$ carries a self-loop — an odd closed walk of length $1$ — so no component is bipartite. Hence $\mathrm{spec}(\hat S)\subseteq(-1,1]$, with $1$ attained by $\tilde{D}^{1/2}\mathbf{1}$.

**Step 3 — consequence.** $\Vert \hat{S}\Vert_2 = 1$, so $\hat{S}$ is a **contraction**: powers converge instead of diverging, and the network is numerically stable at any depth. The price is over-smoothing (Proof 6 of the theory notebook) — but a model that collapses gracefully is far more trainable than one that explodes.

**Illustration on $K_2$.** $A = \begin{pmatrix}0&1\\1&0\end{pmatrix}$, $D = I$:

$$
S_0 = \begin{pmatrix}1&1\\1&1\end{pmatrix}, \ \mathrm{spec} = \{2, 0\}; \qquad \tilde{A} = \begin{pmatrix}1&1\\1&1\end{pmatrix}, \ \tilde{d} = (2,2), \ \hat{S} = \begin{pmatrix}\tfrac12&\tfrac12\\\tfrac12&\tfrac12\end{pmatrix}, \ \mathrm{spec} = \{1, 0\}
$$

Note $\mathrm{spec}(D^{-1/2}AD^{-1/2}) = \{1,-1\}$ for $K_2$: the bare normalized adjacency *does* hit $-1$ (as $K_2$ is bipartite), and the self-loop is exactly what removes it.

$$
\boxed{\mathrm{spec}(S_0) = 2 - \mathrm{spec}(L_{\mathrm{sym}}) \subseteq [0,2] \ \text{(top} = 2); \quad \mathrm{spec}(\hat S) \subseteq (-1,1] \ \text{(contraction)}}
$$

*Key takeaway:* "Add self-loops" is a spectral intervention, not a numerical nicety: it turns an expanding operator into a contraction and destroys bipartiteness, buying stability at the cost of the highest-frequency response.

### Problem L3.3: Which Graphs Over-Smooth Fastest?

Compute $\mu_2(\hat S)$ for (a) $K_n$ and (b) $C_n$, both with self-loops added, and determine how many layers reduce a non-constant mode by half in each case. Interpret.

**Solution**

**(a) Complete graph $K_n$.** With self-loops, $\tilde{A} = J$ and $\tilde{d}_v = n$ for all $v$, so

$$
\hat{S} = \frac{1}{n}J
$$

$J$ has eigenvalues $n$ (once) and $0$ ($n-1$ times), hence

$$
\mathrm{spec}(\hat S) = \{1, \ 0^{(n-1)}\}, \qquad \mu_2 = 0
$$

The collapse is **immediate**: $\hat{S}^{\,1} = \hat{S} = v_1v_1^{\top}$ already, so a *single* propagation step maps every node to the same representation. One layer is the entire budget.

**(b) Cycle $C_n$.** With self-loops, $\tilde{A} = I + A$ and $\tilde{d}_v = 3$, so $\hat{S} = \frac13(I + A)$. The circulant $A$ has eigenvalues $2\cos\frac{2\pi k}{n}$, hence

$$
\mu_k = \frac{1 + 2\cos\frac{2\pi k}{n}}{3}, \qquad k = 0,\dots,n-1
$$

$\mu_0 = 1$ and

$$
\mu_2 = \frac{1 + 2\cos\frac{2\pi}{n}}{3} \approx \frac{3 - \frac{4\pi^2}{n^2}}{3} = 1 - \frac{4\pi^2}{3n^2}
$$

(The most negative is $\mu = -\frac13$ at $k = n/2$, so the rate is governed by $\mu_2$.) For $n = 100$: $\mu_2 \approx 0.99868$.

**Half-lives.**

| Graph | $\mu_2$ | layers for factor $\tfrac12$ |
|---|---|---|
| $K_n$ | $0$ | $1$ |
| $C_{100}$ | $0.99868$ | $\ln(1/2)/\ln(0.99868) \approx 526$ |

**Interpretation.** Over-smoothing speed is governed by the spectral gap $1 - \mu_2$, i.e. by *how well connected* the graph is. Dense and expander-like graphs — social networks, dense citation graphs — collapse within a few layers. Chains, rings, road networks, and molecular graphs have tiny gaps and tolerate much deeper stacks. The universal advice "never go past $3$ layers" is really advice about *well-connected* graphs.

$$
\boxed{\mu_2(K_n) = 0 \ \text{(collapse in 1 layer)}; \quad \mu_2(C_n) \approx 1 - \tfrac{4\pi^2}{3n^2} \ \text{(collapse in } \Theta(n^2) \text{ layers)}}
$$

*Key takeaway:* The same spectral gap that makes a graph a good mixer (Topic 06) makes it a fast over-smoother — good connectivity is simultaneously the friend of diffusion and the enemy of depth.

### Problem L3.4: Why Ng–Jordan–Weiss Normalizes the Rows

Consider a graph with $k$ connected components. Show that the $L_{\mathrm{sym}}$ embedding places the vertices of component $A_i$ along a ray with radius proportional to $\sqrt{d_v}$, rather than at a single point, and prove that normalizing each row to unit length restores the exact point-cluster structure. Illustrate on the star $K_{1,3}$.

**Solution**

**Step 1 — the $L_{\mathrm{sym}}$ kernel.** From $L_{\mathrm{sym}} = D^{1/2}L_{\mathrm{rw}}D^{-1/2}$ we get $L_{\mathrm{sym}}t = 0 \iff L_{\mathrm{rw}}(D^{-1/2}t) = 0$, so

$$
\ker L_{\mathrm{sym}} = D^{1/2}\ker L = \mathrm{span}\big\{D^{1/2}\mathbf{1}_{A_1},\dots,D^{1/2}\mathbf{1}_{A_k}\big\}
$$

**Step 2 — the embedding is a scaled point cluster.** Write the columns of $T = [t_1\cdots t_k]$ in that basis: $T = D^{1/2}[\mathbf{1}_{A_1}\cdots\mathbf{1}_{A_k}]M$ for an invertible $M$. The row of $T$ belonging to $v \in A_i$ is

$$
\tau_v = \sqrt{d_v}\;M_{i,\cdot}
$$

So all vertices of $A_i$ lie on the **ray** spanned by $M_{i,\cdot}$ — same direction, but radius $\sqrt{d_v}$ varying vertex by vertex. Contrast this with $L_{\mathrm{rw}}$ (Problem L3.1), where the radius factor is absent and the cluster is a single point.

**Step 3 — why that breaks $k$-means.** $k$-means minimizes squared *Euclidean* distance, which is sensitive to radius. If component $A_1$ contains both degree-$1$ and degree-$100$ vertices, its embedded points are spread over a factor $10$ in radius, and $k$-means may prefer to split $A_1$ by degree rather than separate $A_1$ from $A_2$ — clustering by degree instead of by structure.

**Step 4 — row normalization is the fix.** Replace $\tau_v$ by $\tau_v/\Vert\tau_v\Vert$. Since $\tau_v = \sqrt{d_v}M_{i,\cdot}$ with $\sqrt{d_v} \gt 0$,

$$
\frac{\tau_v}{\Vert \tau_v\Vert} = \frac{\sqrt{d_v}\,M_{i,\cdot}}{\sqrt{d_v}\,\Vert M_{i,\cdot}\Vert} = \frac{M_{i,\cdot}}{\Vert M_{i,\cdot}\Vert}
$$

which depends only on $i$. All vertices of $A_i$ land on one point of the unit sphere, the $k$ points are distinct (rows of an invertible $M$ are linearly independent, hence not positive multiples of one another), and $k$-means is exact — exactly the situation of Problem L3.1. $\blacksquare$

$$
\boxed{\tau_v = \sqrt{d_v}\,M_{i,\cdot} \ \xrightarrow{\ \text{row normalize}\ } \ M_{i,\cdot}/\Vert M_{i,\cdot}\Vert \ \text{— degree factor removed}}
$$

**Illustration on $K_{1,3}$** (one component, $k = 1$). The kernel vector of $L_{\mathrm{sym}}$ is $D^{1/2}\mathbf{1} \propto (\sqrt3, 1, 1, 1)^{\top}$: the centre sits at radius $\sqrt3 \approx 1.73$ and the leaves at radius $1$, a $73\%$ spread *within a single component*. Row normalization sends all four to the same unit vector, as it must.

*Key takeaway:* Shi–Malik removes the degree factor exactly (via $D^{-1/2}$); Ng–Jordan–Weiss removes it approximately (via row normalization). Skipping either step turns spectral clustering into degree clustering on irregular graphs.